# Resource-Aware Deep Learning: Hands-On

This notebook will teach you, how to track your own (deep-learning) pipeline's resource consumption using tools like [↗CodeCarbon](https://codecarbon.io/), [↗MLFlow](https://mlflow.org/), and the [↗Lamarr Energy Tracker (LET)](https://github.com/lamarr-institute/lamarr-energy-tracker). This tutorial is built on [↗PyTorch](https://pytorch.org) and [↗PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/). Please make sure you have installed and activated the provided virtual environment (either through `(uv) venv` or `conda/mamba`).

We have provided a small Python package `resource_aware_ml` that implements a number of SRResNet architectures inside the `resource_aware_ml.architectures.model` submodule. If you have not yet installed the package, please do so now inside the repository root directory, e.g. using [uv](https://docs.astral.sh/uv/):

```shell-session
$ uv pip install -e .
```

We can now load the `models` submodule, which implements the following architectures in decreasing complexity:
```
SRResNet18
SRResNet10
SRResNet6
SRResNet4
```
![SRResNet Overview](assets/resnet_sc.png)

In [ ]:
from rich import print  # nicer prints

from resource_aware_ml.architectures import models

Choose one of the architectures. Beware that higher complexity will mean more GPU memory consumption. Make sure, you initialise the model; the print function below should print an overview of the architecture.

In [ ]:
# Choose your architecture here
model = models.SRResNet6()

print(model)

Before we go over the training loop, we have to import and initialize a training module. The training module sets up the training, validation, and inference steps of the model, as well as the optimizer. During the training and validation steps it also calls the loss function `loss_fn` that is used to monitor the training process.

In [ ]:
from resource_aware_ml.training.trainer import TrainModule

from torch.optim import AdamW
from torch.nn import L1Loss

In [ ]:
# Change the Optimizer or loss function here.
train_module = TrainModule(model=model, loss_fn=L1Loss(), optimizer=AdamW, lr=1e-3)

We will also need a data module that handles the data loading for us. The data module implemented in the package is designed to load the dataset that is provided in the repository. The data module inherits from the [↗`lightning.LightningDataModule`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.core.LightningDataModule.html) and is simply a collection of dataloaders for the training, validation, test, and inference stages.

In [ ]:
from resource_aware_ml.io.data import H5DataModule

In [ ]:
data_module = H5DataModule(
    data_dir="./data",  # Path to the data in the repository relative to this notebook
    batch_size=20,      # Change if you run out of memory
    fourier=True,       # Our data is in Fourier space
    num_workers=4       # Change the number of concurrent CPU cores loading the data
)

## Task 1: Logging

Implement a logger that logs the experiment data such as the train loss and validation loss to **MLFlow**. Have a look at the lightning docs for more information on [loggers](https://lightning.ai/docs/pytorch/stable/api_references.html#loggers). Save the logger in a variable `logger`.

In [ ]:
# Implement your solution here
from lightning.pytorch.loggers import CSVLogger, MLFlowLogger

logger = [CSVLogger(save_dir="./build"), MLFlowLogger(save_dir="sqlite:///build/mlflow.db")]

## Task 2: Callbacks
Now implement a callback that logs additional metrics and parameters of your model such as:
| Metric                         | Variable Name              |
| ------------------------------ | -------------------------- |
| Number of trainable parameters | `num_trainable_parameters` |
| Total runtime                  | `running_time_total`       |
| Runtime per epoch              | `running_time`             |
| Total power draw               | `power_draw_total`         |
| Power draw per epoch           | `power_draw`               |

| Parameter                      | Variable Name              |
| ---------------------------- | -------------- |
| Model name                   | `model`        |
| Dataset                      | `dataset`      |
| Architecture (GPU Model)     | `architecture` |
| Task (training or inference) | `task`         |



In [ ]:
trainer.model

In [ ]:
import warnings

from lightning.pytorch.callbacks import Callback

class CustomCallback(Callback):

    def __init__(self, *args, **kwargs) -> None:
        self.experiment = None

    def on_fit_end(self, trainer, pl_module):
        if self.experiment is None:
            self._set_up_experiment(trainer)
            self.num_samples = trainer.datamodule.train_length
            self.num_samples += trainer.datamodule.valid_length

        try:
            self._log_metrics()
        except (FileNotFoundError, KeyError) as e:
            warnings.warn(f"{e}. No emissions were logged.", stacklevel=2)

        self._log_params(trainer)

    def on_predict_end(self, trainer, pl_module):
        if self.experiment is None:
            self._set_up_experiment(trainer)
            self.num_samples = trainer.datamodule.predict_length

        try:
            self._log_metrics()
        except (FileNotFoundError, KeyError) as e:
            warnings.warn(f"{e}. No emissions were logged.", stacklevel=2)

        self._log_params(trainer)

    def _set_up_experiment(self, trainer):
        try:
            self.logger = next(
                logger for logger in trainer.loggers if isinstance(logger, MLFlowLogger)
            )
            trainer.carbontracker.tracker.stop()
            self.experiment = self.logger.experiment

        except StopIteration as e:
            raise RuntimeError(
                f"Could not find a MLFlowLogger instance in {trainer.loggers}."
            ) from e

    def _log_metrics(self):
        emission_file = Path(
            self.train_config.logging.codecarbon.output_dir + "/emissions.csv"
        )
        emission_data = pd.read_csv(emission_file).to_dict()

        eval_res = dict(
            running_time_total=emission_data["duration"][0],
            running_time=emission_data["duration"][0] / self.num_samples,
            power_draw_total=emission_data["energy_consumed"][0] * 3.6e6,
            power_draw=emission_data["energy_consumed"][0] * 3.6e6 / self.num_samples,
        )

        for key, val in eval_res.items():
            self.experiment.log_metric(
                key=key,
                value=val,
                run_id=self.logger._run_id,
            )

        self.architecture = emission_data["gpu_model"][0]

        # Remove file after logging all important metrics to mlflow.
        # This prevents codecarbon from creating 'emissions.csv_%d.bak'
        # files in the save directory
        if emission_file.is_file():
            emission_file.unlink()

    def _log_params(self, trainer):
        dataset = trainer.datamodule.data_dir.name

        model = trainer.model.model.__class__.__name__

        params_dict = dict(
            model=model,
            dataset=dataset,
            task=trainer.task,
            architecture=self.architecture,
        )
        for key, val in params_dict.items():
            self.experiment.log_param(
                key=key,
                value=val,
                run_id=self.logger._run_id,
            )


callback = CustomCallback()

Now we can set up the [↗`lightning.Trainer`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.trainer.trainer.Trainer.html). Here, we can set

In [ ]:
from lightning import Trainer

In [ ]:
trainer = Trainer(
    max_epochs=100,
    accelerator="auto",  # Change to gpu, or cpu, if you like
    precision="32-true",
    logger=logger,
    callbacks=callback
)

In [ ]:
trainer.fit(model=train_module)